In [13]:
import numpy as np 
import torch
import pandas as pd
import h5py 
import importlib 
import pickle
from IPython.display import display, Audio
from lightning_scripts import jsinV3DataLoader_precombined_batched as jsinv3

In [3]:
importlib.reload(jsinv3)

MatchedSpeechInNoiseDatasetBatchedRandIx = jsinv3.MatchedSpeechInNoiseDatasetBatchedRandIx

In [5]:
source_h5 = '/mnt/home/jfeather/ceph/data/training_datasets_audio/jsinV3BalancedProcessed/sr_20000/splits/valid_stackedDataframeHDF_n150_VJRUH4IEPDGPNH2JZMULSQKOWYNQ6KMM.pdh5'
noise_h5 = '/mnt/home/jfeather/ceph/data/training_datasets_audio/audioset_dataframes/sr20000/sr20000_balanced_train_segments_raw_exclude_speech_and_only_music.pdh5' 

target_keys = ['signal/word_int', 'signal/speaker_int', 'noise/labels_int']
dataset = MatchedSpeechInNoiseDatasetBatchedRandIx(source_h5, noise_h5, batch_size=2, target_keys=target_keys)

In [19]:
orig_dataset = jsinv3.MatchedSpeechInNoiseDatasetBatched(source_h5, noise_h5, batch_size=2, target_keys=target_keys)

In [20]:
orig_dataset

In [9]:
word_and_speaker_encodings = pickle.load(
    open("/mnt/home/igriffith/ceph/projects/cochdnn/robustness/audio_functions/word_and_speaker_encodings_jsinv3.pckl", "rb")
)
class_map = word_and_speaker_encodings["word_idx_to_word"]
talker_map = word_and_speaker_encodings["speaker_idx_to_speaker"]


In [14]:
np.random.seed(0)

# [output_11, output_12, output_21, output_22], [target_11, target_12, target_21 , target_22] = dataset[0]
audio, labels = dataset[0]

for combined, label_group in zip(audio, labels):
    for ix, example in enumerate(combined):
        for key in label_group.keys():
            label_item = label_group[key][ix]
            if 'word' in key:
                print(f"Word: {class_map[label_item.item()]}")
            elif 'speaker' in key:
                print(f"Speaker: {talker_map[label_item.item()]}")
        display(Audio(example, rate=20_000))

Word: funds
Speaker: 4bi


Word: north
Speaker: fliry-vorru


Word: funds
Speaker: 4bi


Word: north
Speaker: fliry-vorru


Word: cover
Speaker: 46r


Word: better
Speaker: hamiltonstone


Word: cover
Speaker: 46r


Word: better
Speaker: hamiltonstone


In [23]:
## iter test 

def collate_fn(batch):
    batch = batch[0]
    if len(batch) == 2:
        all_audio = []
        for audio in batch[0]:
            all_audio.append(audio.unsqueeze(1))
        labels = []
        for label_set in batch[1]:
            view_labels = {}
            if isinstance(label_set, dict):
                for key, l in label_set.items():
                    view_labels[key] = l.squeeze()
                labels.append(view_labels)
            else:
                labels.append(label_set.squeeze())
        return all_audio, labels 
    elif len(batch) == 4: # no labels
        return [audio.unsqueeze(1) for audio in batch]


dataloader = torch.utils.data.DataLoader(
            dataset,
            batch_size=1,
            num_workers=0, 
            # pin_memory=True,
            # persistent_workers=True,
            shuffle=False,
            collate_fn=collate_fn
        )
orig_dataloader = torch.utils.data.DataLoader(
            orig_dataset,
            batch_size=1,
            num_workers=0, 
            # pin_memory=True,
            # persistent_workers=True,
            shuffle=True,
            collate_fn=collate_fn
        )


In [17]:
from tqdm import tqdm 

for _ in tqdm(dataloader):
    continue

  0%|          | 49/10162 [00:13<45:49,  3.68it/s]


KeyboardInterrupt: 

In [24]:

for _ in tqdm(orig_dataloader):
    continue

  1%|          | 67/10162 [00:18<46:49,  3.59it/s]  


KeyboardInterrupt: 